# 00 · Configuration

Central parameter module imported by all notebooks. Defines dataset paths,
leave-one-dataset-out folds, ablation configurations, the ConvNeXt-Large model
specification with stage-freezing and differential learning rates, MRKR
subsampling, and the curriculum schedule.

## Mount storage

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Mounted.')

Mounted at /content/drive
Mounted.


## Write configuration module

In [2]:
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/Master Thesis')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

config_text = """# Scope 3 Configuration
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Master Thesis")

DATASETS = {
    "oai":      {"png_dir": PROJECT_ROOT/"oai"/"processed",
                 "labels": PROJECT_ROOT/"oai"/"oai_processed_labels.csv", "kl_source":"radiologist"},
    "nhanes3":  {"png_dir": PROJECT_ROOT/"nhanes3"/"processed",
                 "labels": PROJECT_ROOT/"nhanes3"/"labels_split.csv", "kl_source":"radiologist"},
    "mrkr":     {"png_dir": PROJECT_ROOT/"MRKR"/"processed",
                 "labels": PROJECT_ROOT/"MRKR"/"mrkr_labels.csv", "kl_source":"model_predicted"},
    "mendeley": {"png_dir": PROJECT_ROOT/"mendeley_oa"/"processed",
                 "labels": PROJECT_ROOT/"mendeley_oa"/"mendeley_holdout.csv", "kl_source":"radiologist"},
}

MANIFEST_CSV = PROJECT_ROOT/"scope3"/"manifest.csv"
LOCAL_DATA   = Path("/content/local_data")
IMAGES_ZIP   = PROJECT_ROOT/"scope3"/"images.zip"
IMAGES_NPY   = PROJECT_ROOT/"scope3"/"images.npy"      # single packed array (primary loader)
CKPT_DIR     = PROJECT_ROOT/"scope3"/"checkpoints"
RESULTS_DIR  = PROJECT_ROOT/"scope3"/"results"
LOG_DIR      = PROJECT_ROOT/"scope3"/"logs"
for d in [MANIFEST_CSV.parent, CKPT_DIR, RESULTS_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LODO_FOLDS = {
    "fold1_test_mendeley": "mendeley",
    "fold2_test_mrkr":     "mrkr",
    "fold3_test_nhanes3":  "nhanes3",
    "fold4_test_oai":      "oai",
}
SEEDS = [0, 1, 2]

ABLATION_CONFIGS = {
    "A_naive":      {"sampler":False,"noise_loss":False,"curriculum":False,"domain_adv":False,"hierarchical":True,"ordinal":False,"use_quality":False},
    "B_sampler":    {"sampler":True, "noise_loss":False,"curriculum":False,"domain_adv":False,"hierarchical":True,"ordinal":False,"use_quality":False},
    "C_noiseloss":  {"sampler":True, "noise_loss":True, "curriculum":False,"domain_adv":False,"hierarchical":True,"ordinal":False,"use_quality":False},
    "D_curriculum": {"sampler":True, "noise_loss":True, "curriculum":True, "domain_adv":False,"hierarchical":True,"ordinal":False,"use_quality":True},
    "E_domain_adv": {"sampler":True, "noise_loss":True, "curriculum":True, "domain_adv":True, "hierarchical":True,"ordinal":False,"use_quality":True},
}
EXTRA_CONFIGS = {
    "F_direct5class": {"sampler":True,"noise_loss":True,"curriculum":True,"domain_adv":False,"hierarchical":False,"ordinal":False,"use_quality":True},
    "G_ordinal":      {"sampler":True,"noise_loss":True,"curriculum":True,"domain_adv":False,"hierarchical":True, "ordinal":True, "use_quality":True},
    # Combined best: union of components that improved external generalization
    "H_combined":     {"sampler":True,"noise_loss":True,"curriculum":True,"domain_adv":True, "hierarchical":True, "ordinal":True, "use_quality":True},
}
ABLATION_FOLD   = "fold1_test_mendeley"
FULL_METHOD_NAME= "H_combined"
FULL_METHOD     = EXTRA_CONFIGS["H_combined"]

MODEL_NAME   = "convnext_large"
IMG_SIZE     = 224
NUM_CLASSES  = 5
FREEZE_STAGES = 3          # freeze stages 0,1,2 ; train stage 3 + head
LR_HEAD      = 1e-4
LR_BACKBONE  = 1e-5        # 0.1x head
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS= 2
EPOCHS       = 18          # ConvNeXt-L converges ~ep10; early stop handles rest
EARLY_STOP_PATIENCE = 6
BATCH_SIZE   = 32          # ConvNeXt-L memory; use grad-accum if OOM
GRAD_ACCUM   = 2           # effective batch 64
NUM_WORKERS  = 2

MRKR_CAP = 40000           # stratified cap; None = use all

LOSS_WEIGHTS = {"radiologist":1.0, "model_predicted":0.6}

CURRICULUM = {"phase1_end":4, "phase2_end":12, "mrkr_target_weight":0.6}

USE_JOINT_CROP = True
JOINT_CROP_FRAC = 0.80      # keep central 80% (removes borders/labels)

GRL_LAMBDA_MAX = 0.3        # peak adversarial weight; sweep candidates below
GRL_LAMBDA_SWEEP = [0.1, 0.3, 0.5]

SAVE_PREDICTIONS = True

STAGE1_OA_THRESHOLD = 2
USE_TTA = True
TTA_TRANSFORMS = ["identity","hflip","rot+5","rot-5"]

LABEL_QUALITY_CSV = PROJECT_ROOT/"scope3"/"results"/"mrkr_label_quality.csv"
QUALITY_MIN_WEIGHT = 0.2   # floor for low-quality samples

def set_seed(seed):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
"""

p = PROJECT_ROOT/"scope3"/"config.py"
p.parent.mkdir(parents=True, exist_ok=True)
with open(p,"w") as f: f.write(config_text)
print(f"config.py written: {p.stat().st_size:,} bytes")

config.py written: 4,286 bytes


## Verify configuration

In [3]:
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
print('Model      :', config.MODEL_NAME)
print('Freeze<stage:', config.FREEZE_STAGES, '| LR head/bb:', config.LR_HEAD, '/', config.LR_BACKBONE)
print('Batch x accum:', config.BATCH_SIZE, 'x', config.GRAD_ACCUM)
print('MRKR cap   :', config.MRKR_CAP)
print('Configs    :', list(config.ABLATION_CONFIGS.keys()))
print('Full method:', config.FULL_METHOD_NAME)
n_abl=len(config.ABLATION_CONFIGS)*1
n_lodo=len(config.LODO_FOLDS)*len(config.SEEDS)
print(f'Planned runs: {n_abl} ablation(1 seed) + {n_lodo} LODO(3 seeds) + 3 baselines')

Model      : convnext_large
Freeze<stage: 3 | LR head/bb: 0.0001 / 1e-05
Batch x accum: 32 x 2
MRKR cap   : 40000
Configs    : ['A_naive', 'B_sampler', 'C_noiseloss', 'D_curriculum', 'E_domain_adv']
Full method: H_combined
Planned runs: 5 ablation(1 seed) + 12 LODO(3 seeds) + 3 baselines
